In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as msno
from ydata_profiling import ProfileReport
from pathlib import Path

In [ ]:
tickers = ["AAPL", "MSFT", "SPY"]
start_date = "2020-01-01"
end_date = "2025-01-01"

In [ ]:
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)

close_prices = data["Close"].copy()
close_prices.head()

In [ ]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)
close_prices.to_csv("../data/raw/prices.csv")

In [ ]:
returns = close_prices.pct_change().dropna()
returns.head()

In [ ]:
cum_returns = (1 + returns).cumprod() - 1

plt.figure(figsize=(12,6))
for col in cum_returns.columns:
    plt.plot(cum_returns.index, cum_returns[col], label=col)

plt.title("Retornos acumulados")
plt.xlabel("Fecha")
plt.ylabel("Retorno acumulado")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
returns.hist(bins=50, figsize=(14,4), layout=(1, len(returns.columns)))
plt.suptitle("Distribución de retornos diarios")
plt.tight_layout()
plt.show()

In [ ]:
quality_report = pd.DataFrame({
    "n_obs": close_prices.count(),
    "n_missing": close_prices.isna().sum(),
    "pct_missing": close_prices.isna().mean() * 100,
    "n_unique": close_prices.nunique(),
    "min_date": [close_prices[col].dropna().index.min() for col in close_prices.columns],
    "max_date": [close_prices[col].dropna().index.max() for col in close_prices.columns]
})

quality_report

In [ ]:
msno.matrix(close_prices, figsize=(12,4))
plt.show()

In [ ]:
profile = ProfileReport(close_prices.reset_index(), title="Data Quality Report", explorative=True)
profile.to_file("../reports/data_quality_report.html")